# CetaceanNet - Multimodal Attention-Based Acoustic Classification

This notebook walks through the full pipeline for classifying cetacean (whale &
dolphin) species from short audio clips:

1. **Preprocess** raw `.wav` files into fixed-length waveforms.
2. **Extract** three complementary features: MEL spectrogram, Wavelet Scattering
   Transform (WST), and MFCC.
3. **Train** several models, from a custom multimodal attention CNN
   (`CetaceanNet`) to MobileNetV2 baselines.

All reusable logic lives in the `cetacean_net` package under `src/`; this
notebook is just the narrative driver.

## Setup

Install dependencies and (on Colab) mount Google Drive for the dataset.

In [ ]:
!pip install -q -r ../requirements.txt

In [ ]:
# Make the src/ package importable when running from notebooks/.
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

In [ ]:
# On Google Colab, mount Drive to access the dataset.
# from google.colab import drive
# drive.mount('/content/drive')

## 1. Preprocessing & feature extraction

Point `DATA_PATH` at a folder laid out as `DATA_PATH/<species>/<clip>.wav`.

In [ ]:
import numpy as np
import torch

from cetacean_net import (
    AudioPreprocessor,
    CetaceanNet,
    CetaceanNetLight,
    MobileNetV2,
    MobileNetV2Combined,
    build_dataloaders,
    stratified_split,
    train_model,
    visualize_samples,
)

In [ ]:
DATA_PATH = '/content/drive/MyDrive/Data'  # adjust to your dataset location
BATCH_SIZE = 32

In [ ]:
preprocessor = AudioPreprocessor()
X, y = preprocessor.basic_preprocess(DATA_PATH)
print('Waveforms:', X.shape)

In [ ]:
mel_features = preprocessor.compute_mel(X)
wst_features = preprocessor.compute_wst(X)
mfcc_features = preprocessor.compute_mfcc(X)

Build the label mapping and inspect the feature shapes.

In [ ]:
CLASSES = np.unique(y)
NUM_CLASSES = len(CLASSES)
ID2LABEL = {i: category.item() for i, category in enumerate(CLASSES)}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
labels = torch.tensor([LABEL2ID[label] for label in y])

print('ID2LABEL:', ID2LABEL)
print('Num classes:', NUM_CLASSES)
print('MEL :', mel_features.shape)
print('WST :', wst_features.shape)
print('MFCC:', mfcc_features.shape)
print('Labels:', labels.shape)

## 2. Visualise a few feature maps

Sanity-check that extraction produced sensible time-frequency images.

In [ ]:
visualize_samples(mel_features, 'MEL', 'viridis')

In [ ]:
visualize_samples(wst_features, 'WST')

In [ ]:
visualize_samples(mfcc_features, 'MFCC', 'inferno')

## 3. Train / validation split and dataloaders

`build_dataloaders` returns loaders for each single-feature baseline plus the
fused multimodal dataset.

In [ ]:
train_idx, val_idx = stratified_split(labels, test_size=0.2, seed=42)
loaders = build_dataloaders(
    mfcc_features, wst_features, mel_features, labels,
    train_idx, val_idx, batch_size=BATCH_SIZE,
)
train_loader, val_loader = loaders['fusion']

## 4. Train the models

Each call below trains one architecture. Pass `use_wandb=False` to skip
Weights & Biases logging for quick local runs.

### 4a. CetaceanNet - multimodal attention CNN

In [ ]:
cetacean_net = CetaceanNet(num_classes=NUM_CLASSES)
train_model(
    model=cetacean_net,
    train_loader=train_loader, val_loader=val_loader,
    y=y, label2id=LABEL2ID, num_classes=NUM_CLASSES,
    architecture='Multimodal Attention-Based CNN',
    epochs=60, batch_size=BATCH_SIZE, learning_rate=1e-4,
)

### 4b. CetaceanNetLight - depthwise-separable variant

In [ ]:
cetacean_net_light = CetaceanNetLight(num_classes=NUM_CLASSES)
train_model(
    model=cetacean_net_light,
    train_loader=train_loader, val_loader=val_loader,
    y=y, label2id=LABEL2ID, num_classes=NUM_CLASSES,
    architecture='Lightweight CetaceanNet',
    epochs=60, batch_size=BATCH_SIZE, learning_rate=1e-4,
)

### 4c. MobileNetV2 - stacked features (MFCC + WST + MEL as 3 channels)

In [ ]:
mobilenetv2_combined = MobileNetV2Combined(num_classes=NUM_CLASSES)
train_model(
    model=mobilenetv2_combined,
    train_loader=train_loader, val_loader=val_loader,
    y=y, label2id=LABEL2ID, num_classes=NUM_CLASSES,
    architecture='MobileNetV2 (all three features)',
    epochs=60, batch_size=BATCH_SIZE, learning_rate=1e-4,
)

### 4d. MobileNetV2 - single-feature baselines

In [ ]:
train_mel_loader, val_mel_loader = loaders['mel']
mobilenetv2_mel = MobileNetV2(num_classes=NUM_CLASSES)
train_model(
    model=mobilenetv2_mel,
    train_loader=train_mel_loader, val_loader=val_mel_loader,
    y=y, label2id=LABEL2ID, num_classes=NUM_CLASSES,
    architecture='MobileNetV2 (MEL)',
    epochs=60, batch_size=BATCH_SIZE, learning_rate=1e-4,
)

In [ ]:
train_mfcc_loader, val_mfcc_loader = loaders['mfcc']
mobilenetv2_mfcc = MobileNetV2(num_classes=NUM_CLASSES)
train_model(
    model=mobilenetv2_mfcc,
    train_loader=train_mfcc_loader, val_loader=val_mfcc_loader,
    y=y, label2id=LABEL2ID, num_classes=NUM_CLASSES,
    architecture='MobileNetV2 (MFCC)',
    epochs=60, batch_size=BATCH_SIZE, learning_rate=1e-4,
)